In [1]:
import os
import json
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from skimage.draw import polygon
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings('ignore')

# ==================== Device Setup ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cudnn.benchmark = True

# ==================== Hyperparameters ====================
num_epochs = 20
batch_size = 6
learning_rate = 2e-4
weight_decay = 1e-4
patience = 5
num_classes = 2
image_size = 256

# ==================== Dataset ====================
class CocoMaskedDataset(Dataset):
    def __init__(self, images_path, annotations_path, transform=None):
        self.images_path = images_path
        self.transform = transform
        with open(annotations_path, "r") as f:
            coco = json.load(f)
        self.images = {img["id"]: img["file_name"] for img in coco["images"]}
        self.image_ids = list(self.images.keys())
        self.annotations = {}
        for ann in coco["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = np.array(Image.open(os.path.join(self.images_path, self.images[img_id])).convert("RGB"))
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for ann in self.annotations.get(img_id, []):
            for seg in ann.get("segmentation", []):
                if isinstance(seg, list) and len(seg) % 2 == 0:
                    poly_pts = np.array(seg).reshape(-1, 2)
                    rr, cc = polygon(poly_pts[:, 1], poly_pts[:, 0], shape=img.shape[:2])
                    mask[rr, cc] = 1
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]
        return img.float(), mask.long()

# ==================== Augmentations ====================
train_transform = A.Compose([
    A.LongestMaxSize(max_size=image_size),
    A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=0),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.LongestMaxSize(max_size=image_size),
    A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=0),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# ==================== Paths ====================
images_path = r"D:\val2017\val2017"
annotations_path = r"D:\annotations_trainval2017\annotations\instances_val2017.json"

# ==================== Dataset & DataLoader ====================
dataset = CocoMaskedDataset(images_path, annotations_path, transform=train_transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ==================== Model ====================
weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights, aux_loss=True)

# Replace classifier for our num_classes
old_cls = model.classifier
model.classifier = nn.Sequential(
    old_cls[0], old_cls[1], old_cls[2],
    nn.Dropout(0.3),
    nn.Conv2d(256, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128, num_classes, kernel_size=1)
)
model = model.to(device)

# ==================== Loss ====================
class DiceCELoss(nn.Module):
    def __init__(self, weight=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)

    def dice_loss(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        target_one_hot = torch.zeros_like(pred)
        target_one_hot.scatter_(1, target.unsqueeze(1), 1)
        intersection = (pred * target_one_hot).sum(dim=(2,3))
        union = pred.sum(dim=(2,3)) + target_one_hot.sum(dim=(2,3))
        dice = (2*intersection) / (union + 1e-7)
        return 1 - dice.mean()

    def forward(self, pred, target):
        return 0.5*self.ce(pred, target) + 0.5*self.dice_loss(pred, target)

criterion = DiceCELoss(weight=torch.tensor([0.3,0.7]).to(device))
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# ==================== Metrics ====================
def calculate_iou(preds, masks):
    preds = torch.argmax(preds, dim=1)
    intersection = ((preds==1) & (masks==1)).sum().float()
    union = ((preds==1) | (masks==1)).sum().float()
    return (intersection/(union+1e-7)).item()

def calculate_dice(preds, masks):
    preds = torch.argmax(preds, dim=1)
    intersection = ((preds==1) & (masks==1)).sum().float()*2
    total = (preds==1).sum().float() + (masks==1).sum().float()
    return (intersection/(total+1e-7)).item()

# ==================== Training ====================
best_iou = 0.0
patience_counter = 0

for epoch in range(1, num_epochs+1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    # Train
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader, desc="Training"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_loss, ious, dices = 0, [], []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc="Validation"):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)['out']
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            ious.append(calculate_iou(outputs, masks))
            dices.append(calculate_dice(outputs, masks))
    val_loss /= len(val_loader)
    mean_iou = np.mean(ious)
    mean_dice = np.mean(dices)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | IoU: {mean_iou:.4f} | Dice: {mean_dice:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Scheduler
    scheduler.step()

    # Save best
    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(model.state_dict(), "deeplabv3_best.pth")
        print(f"✓ Best model saved (IoU: {best_iou:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"⏹️ Early stopping at epoch {epoch}")
            break

    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda


Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | Memory: 6.44 GB
Train batches: 667 | Val batches: 167

Epoch 1/20


Validation: 100%|████████████████████████████████████████████████████████████████████| 167/167 [01:59<00:00,  1.40it/s]


Train Loss: 0.3211 | Val Loss: 0.2856 | IoU: 0.5976 | Dice: 0.7410 | LR: 0.000200
✓ Best model saved (IoU: 0.5976)

Epoch 2/20


Training:  35%|████████████████████████▋                                             | 235/667 [03:27<06:21,  1.13it/s]


KeyboardInterrupt: 